# Phase 7 — NN (our team) vs expert-features (other team): comparison, latent PCA, report

Source material for the 2-minute video. Everything here is driven by the shared
eval code in `expr_movements.evaluate` and the figures in `expr_movements.viz`,
so the numbers match `expr-evaluate` exactly.

**Run before this notebook** (so the run dirs exist):

```bash
uv run expr-train --config configs/experiment_cnn1d.yaml        # B (NN), LOSO
uv run expr-train --config configs/experiment_cnn1d_intra.yaml  # B (NN), intra-subject
uv run expr-train --config configs/experiment_rf.yaml           # A (classic ML), LOSO
uv run expr-train --config configs/experiment_rf_intra.yaml     # A (classic ML), intra
```

> The expert-features team produces their own run dir via the *same* harness
> (`expr-train` with their config). Point `RUN_A` at it when it exists; the RF
> baseline stands in until then so this notebook runs end-to-end today.

Install the figure deps once: `uv sync --extra viz`.

In [ ]:
from pathlib import Path

from expr_movements.evaluate import (
    compare_protocols,
    compare_runs,
    evaluate_run,
    format_comparison,
    format_protocol_comparison,
)
from expr_movements.viz import latent_pca, plot_confusion_matrix, plot_metric_comparison

# Edit these to your actual run dirs (expr-train prints each one).
RUN_B_LOSO = "outputs/cnn1d_4359693b"   # NN, leave-one-subject-out
RUN_B_INTRA = "outputs/cnn1d_intra"      # NN, intra-subject
RUN_A_LOSO = "outputs/rf_1f7822cf"       # expert-features / classic ML, LOSO

FIGS = Path("outputs/figs")
FIGS.mkdir(parents=True, exist_ok=True)

## 1. A vs B head-to-head (same LOSO split, same eval code)

Course requirement: *extracted features vs expert features*. Same folds, same
metrics — the only difference is the approach.

In [ ]:
ab = compare_runs(RUN_A_LOSO, RUN_B_LOSO, out_path="outputs/comparison.md")
print(format_comparison(ab))

In [ ]:
plot_metric_comparison(ab, FIGS / "compare_clip_level.png", level="clip_level")
plot_metric_comparison(ab, FIGS / "compare_window_level.png", level="window_level")

## 2. Intra-subject vs inter-subject (LOSO) — the subject-dependence gap

Intra-subject is directly comparable to the source paper (Venture 2014, >90%);
LOSO is the real (unseen-person) task. The **gap** between them is the headline
finding: how much harder generalising to an unseen subject is.

In [ ]:
proto = compare_protocols(RUN_B_INTRA, RUN_B_LOSO, out_path="outputs/protocol_comparison.md")
print(format_protocol_comparison(proto))

## 3. Latent PCA — emotion = colour, subject = marker

The interpretability figure. A clean emotion separation that still clusters by
subject is the *visual* form of the subject-dependence story. Read it together
with the held-out Silhouette / Davies-Bouldin numbers (those are the honest
generalisation signal; the PCA encodes the whole dataset with the saved model).

In [ ]:
latent_pca(RUN_B_LOSO, FIGS / "latent_pca.png", title="Multi-task latent (NN)")

## 4. Confusion matrices (the source paper finds Joy the hard class)

In [ ]:
for run, tag in ((RUN_A_LOSO, "A_expert"), (RUN_B_LOSO, "B_nn")):
    s = evaluate_run(run)
    plot_confusion_matrix(
        s["confusion_matrix"]["clip"],
        s["labels"],
        FIGS / f"confusion_{tag}.png",
        title=f"Confusion (clip) — {tag}",
    )

## 5. Why these two approaches (talking points for the video)

- **Expert features (A)**: hand-crafted, interpretable features + classic ML.
  Strong when the right features are known; the source paper's IK-on-12-DOF
  pipeline is exactly this. Interpretability is high, but it bakes in human
  priors about *which* motion cues carry emotion.
- **NN multi-task (B)**: a 1D-CNN encoder learns the representation end-to-end
  under a joint reconstruction + classification objective, so the latent both
  *reconstructs the motion* and *separates the emotions* — which is what makes
  the PCA above meaningful. No hand-picked features, but less directly
  interpretable.
- **What the comparison shows**: fill in from the tables above — which approach
  wins on LOSO Macro-F1, how large the intra→LOSO gap is for each (subject
  dependence), and whether the NN latent separates emotions cleanly or still
  splits by subject.

All figures are in `outputs/figs/`; the markdown reports are
`outputs/comparison.md` and `outputs/protocol_comparison.md`.